# 07_卷积与时间信息

**学习目标：** 理解卷积提取空间特征、时间维保留动作变化、预训练减少从零学习成本。

适合：读过CS50P但基础不熟、刚开始深度学习。按顺序学习；每课先看讲解、跑示例、做练习，再查答案。


## 本课要点

CNN用共享的小卷积核扫描图像。单帧可能分不清拿杯子和喝水，连续帧保留移动过程。R(2+1)D把空间卷积与时间卷积分解；34表示架构层级。先学数据流，再逐层研究复杂网络。


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'tools/learning.py').is_file() and (p / 'config.json').is_file())
sys.path.insert(0, str(ROOT / 'tools'))
import learning as L
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)


## 卷积如何改变通道数


In [ ]:
import torch
from torch import nn
torch.set_num_threads(4)
image = torch.randn(2, 4, 32, 32)
spatial = nn.Conv2d(4, 8, kernel_size=3, padding=1)
feature = spatial(image)
print('输入', image.shape, '特征', feature.shape)
print('参数数量', sum(p.numel() for p in spatial.parameters()))


## 增加时间轴，再聚合到分类分数


In [ ]:
video = torch.randn(2, 4, 8, 32, 32)
tiny_video_model = nn.Sequential(nn.Conv3d(4, 8, 3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool3d(1), nn.Flatten(), nn.Linear(8, 40))
print(video.shape, '→', tiny_video_model(video).shape)
print('仅演示shape，随机参数还没有学会识别动作。')


## 时间顺序与平均的区别


In [ ]:
forward = torch.tensor([0., 1., 2., 3.])
reverse = forward.flip(0)
print('两条序列平均相同：', forward.mean().item(), reverse.mean().item())
print('相邻变化方向不同：', forward.diff().tolist(), reverse.diff().tolist())
print('时间平均会丢失顺序；是否需要更强时序模型必须结合任务与实验。')


## 动手练习

用numel统计tiny_video_model的全部参数数量，填入your_answer。

显示“待完成”表示练习尚未作答；代码示例运行成功不等于练习完成。


In [ ]:
your_answer = None  # TODO：遍历model.parameters()
if your_answer is None:
    print('待完成：统计所有可训练参数的元素数')
else:
    assert your_answer == 1232
    print('练习通过')


## 常见误区

不能用随机模型的一次输出声称准确率；新增IR通道不等于换成四种传感器模态。


## 过关检查与选读

解释空间与时间信息；预训练/微调/最终权重区别；说出4通道来源。

选做：改变示例中一个参数，先预测现象再运行，并记下原因。

[对应参考答案](../参考答案/07_卷积与时间信息.ipynb) · [官方选读](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)

完成后在README.md#学习记录写下一句自己的解释。
